# Notebook 3: L2 Regularization — Ridge Regression

**Difficulty:** ⭐⭐⭐⭐ | **Estimated Time:** 3 hours  
**Theme:** California-style House Price Prediction  
**Week 8 — Regularization, Hyperparameter Tuning & Optimization**

---

## What You Will Learn
1. Why ordinary least squares (OLS) breaks down when features are correlated
2. How L2 regularization (Ridge) solves the multicollinearity problem
3. The analytical derivation and its Bayesian interpretation
4. How to implement Ridge from scratch and verify with scikit-learn
5. How to tune α using cross-validation

---

## Real-World Analogy First

> **The Highway Speed Limit**
>
> Imagine a government that wants to reduce traffic accidents on a highway. It has two options:
> - **Option A (Lasso / L1):** Ban certain cars entirely — some cars go at full speed, others must park.
> - **Option B (Ridge / L2):** Set a speed limit — ALL cars can still drive, but nobody goes too fast.
>
> Ridge regression is Option B. It does **not** force any coefficient to exactly zero (no car is banned). Instead, it proportionally shrinks **all** coefficients — every feature still contributes to the prediction, just in moderated amounts.
>
> In house price prediction: `sqft` and `num_rooms` are correlated. OLS might assign `+200` to `sqft` and `-190` to `num_rooms` — two cars racing in opposite directions, nearly cancelling out. Ridge says: "slow down — assign `+30` to each." The prediction is equally accurate but the model is far more stable.

---

## Why L2 Regularization Matters

| Problem | OLS Behaviour | Ridge Solution |
|---------|--------------|----------------|
| Correlated features (`sqft` ↔ `rooms`) | Huge opposite-sign coefficients | Distributes weight evenly |
| Near-singular `XᵀX` matrix | Numerically unstable / no solution | `XᵀX + αI` always invertible |
| High-variance predictions | Overfits training data | Reduces variance (bias-variance tradeoff) |
| All features relevant | OLS is fine but fragile | Ridge is robust with minimal bias cost |

**Rule of thumb:** Use Ridge when you believe *all* features are genuinely relevant but some are correlated. Use Lasso when you suspect many features are irrelevant noise.

## The Mathematics of Ridge Regression

### Objective Function

$$\text{Minimize} \quad J(\theta) = \frac{1}{2n} \sum_{i=1}^{n}(y_i - \hat{y}_i)^2 + \alpha \sum_{j=1}^{p} \theta_j^2$$

- The first term is the familiar Mean Squared Error (MSE)
- The second term $\alpha \sum \theta_j^2$ is the **L2 penalty** — it punishes large coefficients
- $\alpha \geq 0$ controls the trade-off: $\alpha = 0$ is pure OLS, larger $\alpha$ shrinks more
- **Note:** The intercept $\theta_0$ is typically NOT penalized (we center the data)

### Analytical (Closed-Form) Solution

Taking the gradient and setting to zero:

$$\theta^* = (X^\top X + \alpha I)^{-1} X^\top y$$

**Why does $\alpha I$ matter?**  
- $X^\top X$ can be singular (non-invertible) when features are correlated
- Adding $\alpha I$ (a positive diagonal term) makes the matrix **always invertible**
- This is called **Tikhonov regularization** in numerical linear algebra
- Condition number $\kappa(X^\top X + \alpha I) < \kappa(X^\top X)$ for any $\alpha > 0$

### Limiting Behaviour
- As $\alpha \to 0$: Ridge $\to$ OLS (no regularization)
- As $\alpha \to \infty$: $\theta \to \mathbf{0}$ (all coefficients shrink to zero)
- Unlike Lasso, Ridge **never** reaches exactly zero for finite $\alpha$

### Geometric Interpretation

Ridge regression can be viewed as a **constrained optimization**:

$$\text{Minimize MSE subject to} \sum_{j=1}^{p} \theta_j^2 \leq t$$

The constraint region $\sum \theta_j^2 \leq t$ is a **circle** (sphere in higher dimensions).  
Because the circle has no corners (unlike the L1 diamond), the minimum is rarely on a coordinate axis — so coefficients are shrunk but **not zeroed out**.

### Bayesian Interpretation

Ridge regression equals **Maximum A Posteriori (MAP) estimation** with a Gaussian prior:

$$\theta_j \sim \mathcal{N}\left(0, \frac{\sigma^2}{\alpha}\right)$$

- **Prior says:** "Each coefficient is probably near zero; I have uncertainty $\sigma^2/\alpha$"
- **Larger $\alpha$** = tighter (more certain) prior = stronger pull toward zero
- **Smaller $\alpha$** = wider prior = almost no constraint = approaches OLS
- Plain English: Ridge is what happens when you tell the algorithm "I believe most coefficients are probably small — prove me wrong with the data."

In [ ]:
# ============================================================
# CELL 1: Imports and Global Settings
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.linear_model import Ridge, LinearRegression, RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error

# Reproducibility
np.random.seed(42)

# Seaborn theme — applied globally
sns.set_theme(style='whitegrid')

# Consistent colour palette
PALETTE = sns.color_palette('husl', 8)
RIDGE_COLOR = '#2196F3'   # blue
OLS_COLOR   = '#F44336'   # red
TRUTH_COLOR = '#4CAF50'   # green

print('Libraries loaded. NumPy seed set to 42.')
print(f'NumPy {np.__version__} | Pandas {pd.__version__}')

In [ ]:
# ============================================================
# CELL 2: Generate Synthetic California-Style House Price Data
# 300 samples, 20 features including correlated pairs
# ============================================================
np.random.seed(42)
n_samples = 300

# --- Independent base features ---
sqft          = np.random.normal(1800, 500, n_samples).clip(500, 4500)
age           = np.random.uniform(1, 50, n_samples)
lot_size      = np.random.normal(6000, 1500, n_samples).clip(2000, 15000)
garage        = np.random.randint(0, 4, n_samples).astype(float)
bathrooms     = np.random.randint(1, 5, n_samples).astype(float)
stories       = np.random.randint(1, 4, n_samples).astype(float)
pool          = np.random.binomial(1, 0.2, n_samples).astype(float)
school_rating = np.random.uniform(3, 10, n_samples)
crime_index   = np.random.uniform(1, 8, n_samples)
distance_cbd  = np.random.uniform(1, 30, n_samples)

# --- Correlated pairs (the multicollinearity problem) ---
# num_rooms is highly correlated with sqft (r ~ 0.95)
num_rooms = 0.003 * sqft + np.random.normal(0, 0.5, n_samples)
num_rooms = num_rooms.clip(2, 12)

# renovation_year is correlated with age (newer house = recently renovated)
renovation_year = 2024 - age + np.random.normal(0, 3, n_samples)

# zip_score correlated with school_rating and inversely with crime
zip_score = 0.4 * school_rating - 0.3 * crime_index + np.random.normal(0, 0.5, n_samples)

# neighbourhood_price is correlated with distance_cbd (closer = pricier)
neighbourhood_price = 1000 - 20 * distance_cbd + np.random.normal(0, 50, n_samples)

# --- Noise / less-informative features ---
month_listed  = np.random.randint(1, 13, n_samples).astype(float)
agent_id      = np.random.randint(1, 50, n_samples).astype(float)   # irrelevant
listing_photos = np.random.randint(5, 40, n_samples).astype(float)  # weakly relevant
basement      = np.random.binomial(1, 0.4, n_samples).astype(float)
fireplace     = np.random.binomial(1, 0.35, n_samples).astype(float)

# --- True price (ground truth relationship) ---
price = (
    100_000
    + 120  * sqft
    - 800  * age
    + 4    * lot_size
    + 12_000 * garage
    + 15_000 * bathrooms
    + 8_000  * stories
    + 25_000 * pool
    + 10_000 * school_rating
    - 8_000  * crime_index
    - 5_000  * distance_cbd
    + 5_000  * num_rooms       # correlated with sqft!
    + 500    * renovation_year
    + 3_000  * zip_score
    + 200    * neighbourhood_price
    + 2_000  * listing_photos
    + 10_000 * basement
    + 7_000  * fireplace
    + np.random.normal(0, 30_000, n_samples)   # noise
)

# Assemble DataFrame
feature_names = [
    'sqft', 'age', 'lot_size', 'garage', 'bathrooms', 'stories',
    'pool', 'school_rating', 'crime_index', 'distance_cbd',
    'num_rooms', 'renovation_year', 'zip_score', 'neighbourhood_price',
    'month_listed', 'agent_id', 'listing_photos', 'basement',
    'fireplace', 'sqft_x_rooms'  # interaction term (extra collinearity)
]

# Add an interaction/redundant feature
sqft_x_rooms = sqft * num_rooms / 1000

X_df = pd.DataFrame({
    'sqft': sqft, 'age': age, 'lot_size': lot_size,
    'garage': garage, 'bathrooms': bathrooms, 'stories': stories,
    'pool': pool, 'school_rating': school_rating, 'crime_index': crime_index,
    'distance_cbd': distance_cbd, 'num_rooms': num_rooms,
    'renovation_year': renovation_year, 'zip_score': zip_score,
    'neighbourhood_price': neighbourhood_price, 'month_listed': month_listed,
    'agent_id': agent_id, 'listing_photos': listing_photos,
    'basement': basement, 'fireplace': fireplace, 'sqft_x_rooms': sqft_x_rooms
})

y_series = pd.Series(price, name='price')

print(f'Dataset shape: {X_df.shape}')
print(f'Price range: ${price.min():,.0f} – ${price.max():,.0f}')
print(f'\nCorrelation of sqft <-> num_rooms: {np.corrcoef(sqft, num_rooms)[0,1]:.3f}')
print(f'Correlation of age  <-> renovation_year: {np.corrcoef(age, renovation_year)[0,1]:.3f}')
X_df.describe().round(2)

In [ ]:
# ============================================================
# CELL 3: Visualise Key Correlations in the Dataset
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1) Correlation heatmap of all features
corr_matrix = X_df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, ax=axes[0], mask=mask,
    cmap='coolwarm', vmin=-1, vmax=1,
    annot=False, linewidths=0.4, cbar_kws={'shrink': 0.7}
)
axes[0].set_title('Feature Correlation Matrix', fontsize=12, fontweight='bold')
axes[0].tick_params(axis='x', rotation=90, labelsize=7)
axes[0].tick_params(axis='y', labelsize=7)

# 2) Scatter: sqft vs num_rooms (the correlated pair)
axes[1].scatter(sqft, num_rooms, alpha=0.4, color=RIDGE_COLOR, s=20)
axes[1].set_xlabel('sqft')
axes[1].set_ylabel('num_rooms')
r = np.corrcoef(sqft, num_rooms)[0, 1]
axes[1].set_title(f'sqft vs num_rooms  (r = {r:.3f})', fontsize=12, fontweight='bold')
# Annotate the multicollinearity problem
axes[1].text(0.05, 0.92, 'HIGH correlation = multicollinearity risk',
             transform=axes[1].transAxes, fontsize=9, color='darkred')

# 3) Price distribution
axes[2].hist(price / 1000, bins=30, color=TRUTH_COLOR, edgecolor='white', alpha=0.8)
axes[2].set_xlabel('Price ($000s)')
axes[2].set_ylabel('Count')
axes[2].set_title('House Price Distribution', fontsize=12, fontweight='bold')
axes[2].axvline(np.median(price) / 1000, color='red', linestyle='--', label=f'Median ${np.median(price)/1000:.0f}k')
axes[2].legend()

plt.tight_layout()
plt.suptitle('Dataset Overview: California House Prices', fontsize=14, fontweight='bold', y=1.02)
plt.savefig('ridge_dataset_overview.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure saved: ridge_dataset_overview.png')

In [ ]:
# ============================================================
# CELL 4: Ridge From Scratch — Analytical Solution
# theta = (X^T X + alpha * I)^{-1} X^T y
# ============================================================

def ridge_analytical(X, y, alpha=1.0):
    """
    Compute Ridge regression coefficients using the closed-form solution.

    Parameters
    ----------
    X     : (n, p) design matrix (should include intercept column or be pre-centred)
    y     : (n,)  target vector
    alpha : float, regularisation strength (>=0)

    Returns
    -------
    theta : (p,) coefficient vector
    """
    n, p = X.shape
    # np.linalg.solve is preferred over inv() for numerical stability
    # Solves the linear system (X^T X + alpha I) theta = X^T y
    A = X.T @ X + alpha * np.eye(p)   # (p x p) matrix — always positive definite
    b = X.T @ y                        # (p,) right-hand side
    theta = np.linalg.solve(A, b)      # more stable than np.linalg.inv(A) @ b
    return theta


# --- Prepare data: scale features, mean-centre target ---
X_raw = X_df.values
y_raw = y_series.values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)   # zero mean, unit variance
y_centred = y_raw - y_raw.mean()         # centre target to skip intercept in penalty

# --- Fit with our scratch implementation ---
ALPHA_DEMO = 10.0
theta_scratch = ridge_analytical(X_scaled, y_centred, alpha=ALPHA_DEMO)

# --- Fit with sklearn Ridge for verification ---
# fit_intercept=False because we manually centred the data
sk_ridge = Ridge(alpha=ALPHA_DEMO, fit_intercept=False)
sk_ridge.fit(X_scaled, y_centred)
theta_sklearn = sk_ridge.coef_

# --- Compare ---
comparison_df = pd.DataFrame({
    'Feature': feature_names,
    'Scratch': theta_scratch,
    'Sklearn': theta_sklearn,
    'Difference': np.abs(theta_scratch - theta_sklearn)
})

print(f'Ridge (alpha={ALPHA_DEMO}) — Scratch vs Sklearn Comparison')
print('=' * 60)
print(comparison_df.to_string(index=False, float_format='{:.6f}'.format))
print(f'\nMax absolute difference: {comparison_df["Difference"].max():.2e}')
print('Verification: PASSED' if comparison_df['Difference'].max() < 1e-6 else 'Verification: FAILED')

In [ ]:
# ============================================================
# CELL 5: Ridge vs OLS Coefficients — Side-by-Side Bar Chart
# Demonstrates how Ridge moderates extreme OLS coefficients
# ============================================================

# OLS solution (alpha = 0 analytically)
try:
    theta_ols = np.linalg.solve(X_scaled.T @ X_scaled, X_scaled.T @ y_centred)
    ols_stable = True
except np.linalg.LinAlgError:
    theta_ols = np.linalg.lstsq(X_scaled, y_centred, rcond=None)[0]
    ols_stable = False

theta_ridge_demo = ridge_analytical(X_scaled, y_centred, alpha=ALPHA_DEMO)

x_pos = np.arange(len(feature_names))
width = 0.4

fig, ax = plt.subplots(figsize=(14, 5))

bars_ols   = ax.bar(x_pos - width/2, theta_ols,         width=width, label='OLS (α=0)',   color=OLS_COLOR,   alpha=0.8)
bars_ridge = ax.bar(x_pos + width/2, theta_ridge_demo,  width=width, label=f'Ridge (α={ALPHA_DEMO})', color=RIDGE_COLOR, alpha=0.8)

ax.axhline(0, color='black', linewidth=0.8, linestyle='-')
ax.set_xticks(x_pos)
ax.set_xticklabels(feature_names, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Standardised Coefficient', fontsize=11)
ax.set_title('OLS vs Ridge Coefficients\n(Note: large opposite-sign OLS coefficients for correlated features)', fontsize=12, fontweight='bold')
ax.legend(fontsize=11)

# Annotate the correlated pair
for idx, fname in enumerate(feature_names):
    if fname in ['sqft', 'num_rooms']:
        ax.annotate('correlated\npair', xy=(idx, max(theta_ols[idx], 0) + 500),
                    fontsize=8, color='darkred', ha='center')

plt.tight_layout()
plt.savefig('ridge_vs_ols_coefficients.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'\nOLS sqft coefficient:      {theta_ols[0]:>10.2f}')
print(f'OLS num_rooms coefficient: {theta_ols[10]:>10.2f}')
print(f'Ridge sqft coefficient:    {theta_ridge_demo[0]:>10.2f}')
print(f'Ridge num_rooms coeff:     {theta_ridge_demo[10]:>10.2f}')
print('\nObservation: Ridge distributes weight more evenly between correlated features.')

In [ ]:
# ============================================================
# CELL 6: Regularisation Path — Coefficients vs log(alpha)
# Each line = one feature. Shows how coefficients shrink to zero.
# ============================================================

# Range of alpha values: logarithmically spaced from 0.001 to 10000
alphas = np.logspace(-3, 4, 100)

# Store coefficient for each (feature, alpha) pair
coef_paths = np.zeros((len(alphas), len(feature_names)))

for i, alpha in enumerate(alphas):
    coef_paths[i, :] = ridge_analytical(X_scaled, y_centred, alpha=alpha)

fig, ax = plt.subplots(figsize=(13, 6))

# Use a qualitative palette with enough distinct colours
palette = plt.cm.tab20(np.linspace(0, 1, len(feature_names)))

for j, fname in enumerate(feature_names):
    ax.plot(np.log10(alphas), coef_paths[:, j],
            label=fname, color=palette[j], linewidth=1.5, alpha=0.85)

# Highlight the demo alpha
ax.axvline(np.log10(ALPHA_DEMO), color='black', linestyle='--', linewidth=1.2, label=f'Demo α={ALPHA_DEMO}')
ax.axhline(0, color='gray', linewidth=0.7, linestyle=':')

ax.set_xlabel('log₁₀(α)  — Regularisation Strength →', fontsize=12)
ax.set_ylabel('Standardised Coefficient', fontsize=12)
ax.set_title('Ridge Regularisation Path\n(All coefficients shrink smoothly toward 0 — never exactly 0)', fontsize=12, fontweight='bold')

# Legend outside the plot
ax.legend(loc='upper right', bbox_to_anchor=(1.22, 1.0),
          fontsize=8, framealpha=0.8, title='Feature', title_fontsize=9)

# Annotate extremes
ax.text(-2.8, coef_paths[0, :].max() * 0.85, 'OLS\n(α→0)', fontsize=9, color='navy', ha='left')
ax.text(3.7, 300, 'Over-regularised\n(α→∞, θ→0)', fontsize=9, color='darkred', ha='right')

plt.tight_layout()
plt.savefig('ridge_regularisation_path.png', dpi=120, bbox_inches='tight')
plt.show()
print('Key insight: ALL coefficients shrink smoothly — Ridge never produces exact zeros.')

In [ ]:
# ============================================================
# CELL 7: Multicollinearity Demo
# Show OLS instability vs Ridge stability when features are correlated
# Method: Bootstrap — resample the data 200 times, fit both models,
# compare the variance of coefficient estimates
# ============================================================

# Use only the highly correlated pair for clarity
# sqft (index 0) and num_rooms (index 10)
X_small = X_scaled[:, [0, 10]]   # sqft and num_rooms only
y_small = y_centred

n_bootstrap = 300
ols_coefs   = np.zeros((n_bootstrap, 2))
ridge_coefs = np.zeros((n_bootstrap, 2))
ALPHA_STABLE = 50.0

for b in range(n_bootstrap):
    # Resample with replacement
    idx = np.random.choice(n_samples, size=n_samples, replace=True)
    Xb, yb = X_small[idx], y_small[idx]

    # OLS (may be unstable with correlated features)
    try:
        ols_coefs[b] = np.linalg.solve(Xb.T @ Xb, Xb.T @ yb)
    except np.linalg.LinAlgError:
        ols_coefs[b] = np.linalg.lstsq(Xb, yb, rcond=None)[0]

    # Ridge
    ridge_coefs[b] = ridge_analytical(Xb, yb, alpha=ALPHA_STABLE)

# --- Visualise ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax_i, (coefs, label, color) in enumerate(
        [(ols_coefs, 'OLS', OLS_COLOR), (ridge_coefs, f'Ridge (α={ALPHA_STABLE})', RIDGE_COLOR)]):
    ax = axes[ax_i]
    ax.scatter(coefs[:, 0], coefs[:, 1], alpha=0.25, s=15, color=color)
    # Draw confidence ellipse (mean ± 2 std)
    ax.errorbar(coefs[:, 0].mean(), coefs[:, 1].mean(),
                xerr=2*coefs[:, 0].std(), yerr=2*coefs[:, 1].std(),
                fmt='o', color='black', capsize=5, markersize=8, linewidth=2, zorder=5,
                label=f'Mean ± 2σ')
    ax.set_xlabel('θ_sqft', fontsize=11)
    ax.set_ylabel('θ_num_rooms', fontsize=11)
    ax.set_title(f'{label} — Bootstrap Coefficient Distribution\n'
                 f'σ_sqft={coefs[:,0].std():.1f}, σ_rooms={coefs[:,1].std():.1f}',
                 fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)

plt.suptitle('Multicollinearity: OLS instability vs Ridge stability (300 bootstrap samples)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('ridge_bootstrap_stability.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'OLS   θ_sqft std:      {ols_coefs[:,0].std():>8.2f}')
print(f'Ridge θ_sqft std:      {ridge_coefs[:,0].std():>8.2f}')
print(f'Variance reduction:    {(1 - ridge_coefs[:,0].std()/ols_coefs[:,0].std())*100:.1f}%')
print('\nConclusion: Ridge has dramatically lower coefficient variance — more trustworthy estimates.')

In [ ]:
# ============================================================
# CELL 8: Condition Number Analysis
# kappa(X^T X) vs kappa(X^T X + alpha * I)
# Condition number measures how sensitive a matrix solution is to noise
# High kappa => numerically unstable
# ============================================================

alphas_cond = np.logspace(-3, 4, 80)
XtX = X_scaled.T @ X_scaled

kappa_xtx     = np.linalg.cond(XtX)                 # condition number of raw X^T X
kappa_ridge   = [np.linalg.cond(XtX + a * np.eye(XtX.shape[0])) for a in alphas_cond]

fig, ax = plt.subplots(figsize=(10, 5))

ax.semilogx(alphas_cond, kappa_ridge, color=RIDGE_COLOR, linewidth=2.5, label='κ(XᵀX + αI)')
ax.axhline(kappa_xtx, color=OLS_COLOR, linewidth=2, linestyle='--', label=f'κ(XᵀX) = {kappa_xtx:.1e}  [OLS baseline]')
ax.axhline(1e4, color='goldenrod', linewidth=1.5, linestyle=':', label='Practical threshold (1e4)')

ax.fill_between(alphas_cond, kappa_ridge, kappa_xtx, alpha=0.12, color=RIDGE_COLOR)

ax.set_xlabel('Regularisation Strength α', fontsize=12)
ax.set_ylabel('Condition Number κ  (log scale)', fontsize=12)
ax.set_yscale('log')
ax.set_title('Ridge Regularisation Improves Numerical Conditioning\n'
             'Lower condition number = more stable, less sensitive to data perturbations',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)

# Annotate the OLS instability region
ax.text(alphas_cond[0] * 1.5, kappa_xtx * 0.6,
        f'OLS κ = {kappa_xtx:.1e}\n(potentially unstable)', fontsize=9, color='darkred')

plt.tight_layout()
plt.savefig('ridge_condition_number.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Condition number of X^T X alone:       {kappa_xtx:.4e}')
print(f'Condition number at alpha=1:            {np.linalg.cond(XtX + 1.0*np.eye(20)):.4e}')
print(f'Condition number at alpha=100:          {np.linalg.cond(XtX + 100.0*np.eye(20)):.4e}')
print('Adding alpha*I is like adding a stabiliser — the larger alpha, the more stable.')

In [ ]:
# ============================================================
# CELL 9: Bayesian Interpretation — Gaussian Prior Visualisation
# Ridge == MAP with N(0, sigma^2 / alpha) prior on each coefficient
# Larger alpha => tighter prior => stronger shrinkage toward 0
# ============================================================

from scipy.stats import norm

theta_range = np.linspace(-4, 4, 500)

# alpha values to show: small alpha = flat prior (weak), large alpha = tight prior (strong)
alpha_prior = [0.1, 1.0, 5.0, 20.0, 100.0]
sigma_sq    = 1.0   # noise variance (normalised)

fig, ax = plt.subplots(figsize=(10, 5))

colors_prior = plt.cm.plasma(np.linspace(0.15, 0.85, len(alpha_prior)))

for alpha_p, col in zip(alpha_prior, colors_prior):
    prior_std = np.sqrt(sigma_sq / alpha_p)
    pdf       = norm.pdf(theta_range, loc=0, scale=prior_std)
    ax.plot(theta_range, pdf, color=col, linewidth=2.2,
            label=f'α={alpha_p:>6}  →  σ_prior={prior_std:.2f}')
    # Fill to show the credible region
    ax.fill_between(theta_range, pdf, alpha=0.06, color=col)

ax.axvline(0, color='black', linestyle='--', linewidth=1, alpha=0.5, label='Prior mean = 0')
ax.set_xlabel('Coefficient value θ', fontsize=12)
ax.set_ylabel('Prior probability density', fontsize=12)
ax.set_title('Bayesian Interpretation of Ridge Regression\n'
             'Ridge = MAP estimation with Gaussian prior N(0, σ²/α) on each coefficient',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=9, title='Regularisation α', title_fontsize=10)

# Annotation box
ax.text(0.55, 0.97,
        'Larger α → tighter prior\n→ stronger belief θ ≈ 0\n→ more shrinkage',
        transform=ax.transAxes, fontsize=10, va='top',
        bbox=dict(boxstyle='round,pad=0.4', fc='lightyellow', ec='gray', alpha=0.9))

plt.tight_layout()
plt.savefig('ridge_bayesian_prior.png', dpi=120, bbox_inches='tight')
plt.show()

print('Bayesian view: When you increase alpha, you tell the model')
print('  "I am more certain that the true coefficients are near zero."')
print('  The data must then work harder to move coefficients away from 0.')

In [ ]:
# ============================================================
# CELL 10: Cross-Validation — Find the Optimal alpha
# Compare OLS vs Ridge at various alpha using 5-fold CV MSE
# ============================================================

alphas_cv = np.logspace(-2, 4, 60)
cv        = KFold(n_splits=5, shuffle=True, random_state=42)

cv_mse_ridge  = []
cv_mse_std    = []

for alpha in alphas_cv:
    # Pipeline: scale then ridge (scale inside CV to avoid data leakage)
    pipe = Pipeline([('scaler', StandardScaler()),
                     ('ridge', Ridge(alpha=alpha, fit_intercept=True))])
    scores = cross_val_score(pipe, X_raw, y_raw,
                             cv=cv, scoring='neg_mean_squared_error')
    cv_mse_ridge.append(-scores.mean())
    cv_mse_std.append(scores.std())

cv_mse_ridge = np.array(cv_mse_ridge)
cv_mse_std   = np.array(cv_mse_std)

# OLS baseline (alpha=0 equivalent)
pipe_ols = Pipeline([('scaler', StandardScaler()),
                     ('ols', LinearRegression())])
ols_cv_scores = cross_val_score(pipe_ols, X_raw, y_raw,
                                cv=cv, scoring='neg_mean_squared_error')
ols_cv_mse = -ols_cv_scores.mean()

# Best alpha
best_idx   = np.argmin(cv_mse_ridge)
best_alpha = alphas_cv[best_idx]
best_mse   = cv_mse_ridge[best_idx]

# --- Plot ---
fig, ax = plt.subplots(figsize=(11, 5))

ax.semilogx(alphas_cv, cv_mse_ridge / 1e9, color=RIDGE_COLOR, linewidth=2.5, label='Ridge CV MSE')
ax.fill_between(alphas_cv,
                (cv_mse_ridge - cv_mse_std) / 1e9,
                (cv_mse_ridge + cv_mse_std) / 1e9,
                alpha=0.15, color=RIDGE_COLOR, label='± 1 std')
ax.axhline(ols_cv_mse / 1e9, color=OLS_COLOR, linewidth=2,
           linestyle='--', label=f'OLS CV MSE = {ols_cv_mse/1e9:.3f}B')
ax.axvline(best_alpha, color='black', linestyle=':', linewidth=1.5,
           label=f'Best α = {best_alpha:.2f}  (MSE={best_mse/1e9:.3f}B)')

ax.scatter([best_alpha], [best_mse / 1e9], s=100, color='gold',
           zorder=5, edgecolors='black', linewidth=1.5)

ax.set_xlabel('α (Regularisation Strength) — log scale', fontsize=12)
ax.set_ylabel('CV Mean Squared Error (×10⁹)', fontsize=12)
ax.set_title('5-Fold CV: Ridge MSE vs α\n(Optimal Ridge < OLS — regularisation helps)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('ridge_cv_curve.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Best alpha:     {best_alpha:.4f}')
print(f'Ridge best MSE: {best_mse:.4e}')
print(f'OLS CV MSE:     {ols_cv_mse:.4e}')
print(f'Improvement:    {(1 - best_mse/ols_cv_mse)*100:.2f}%')

In [ ]:
# ============================================================
# CELL 11: Geometric Interpretation — L2 Constraint Region
# 2D visualisation: OLS contours + L2 circle constraint
# Shows why Ridge shrinks but doesn't zero out coefficients
# ============================================================

# Use a 2D toy example with the correlated pair (sqft, num_rooms)
# to visualise the geometry clearly
X_geo = X_scaled[:, [0, 10]]    # sqft, num_rooms
y_geo = y_centred

# OLS solution
theta_ols_geo = np.linalg.lstsq(X_geo, y_geo, rcond=None)[0]

# Build a grid over theta space to plot MSE contours
t1 = np.linspace(-3e4, 3e4, 300)
t2 = np.linspace(-3e4, 3e4, 300)
T1, T2 = np.meshgrid(t1, t2)

# MSE as a function of (theta1, theta2)
n_geo = X_geo.shape[0]
MSE_grid = np.zeros_like(T1)
for i in range(len(t1)):
    theta_mat = np.stack([T1[:, i], T2[:, i]], axis=1)  # (300, 2)
    residuals = y_geo[:, None] - X_geo @ theta_mat.T    # (n, 300)
    MSE_grid[:, i] = (residuals ** 2).mean(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax_i, alpha_g in enumerate([0, 1e3]):
    ax = axes[ax_i]
    # Draw MSE contours
    levels = np.percentile(MSE_grid, [10, 20, 35, 55, 75, 90])
    ax.contour(T1, T2, MSE_grid, levels=levels, cmap='Blues', alpha=0.7, linewidths=1.2)

    if alpha_g > 0:
        # Draw L2 constraint circle
        # Radius = constraint budget t
        radius = np.sqrt(np.sum(ridge_analytical(X_geo, y_geo, alpha=alpha_g)**2)) * 1.5
        circle = plt.Circle((0, 0), radius, color=RIDGE_COLOR, fill=False,
                             linewidth=2.5, linestyle='--', label=f'L2 constraint (r={radius:.0f})')
        ax.add_patch(circle)
        theta_constr = ridge_analytical(X_geo, y_geo, alpha=alpha_g)
        ax.scatter(*theta_constr, s=120, color=RIDGE_COLOR, zorder=5,
                   marker='D', label=f'Ridge solution')
        ax.set_title('Ridge (L2): solution pulled toward\ncircle — not to an axis (no zeros)',
                     fontsize=11, fontweight='bold')
    else:
        ax.set_title('OLS: unconstrained minimum',
                     fontsize=11, fontweight='bold')

    # OLS solution marker
    ax.scatter(*theta_ols_geo, s=150, color=OLS_COLOR, zorder=5,
               marker='*', label='OLS solution')

    ax.axhline(0, color='black', linewidth=0.6)
    ax.axvline(0, color='black', linewidth=0.6)
    ax.set_xlabel('θ_sqft', fontsize=11)
    ax.set_ylabel('θ_num_rooms', fontsize=11)
    ax.legend(fontsize=9)
    ax.set_xlim(-3e4, 3e4)
    ax.set_ylim(-3e4, 3e4)

plt.suptitle('Geometric Interpretation: L2 Constraint Region is a Circle (No Corners → No Sparsity)',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('ridge_geometry.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# CELL 12: sklearn RidgeCV — Efficient alpha selection
# RidgeCV uses generalised cross-validation (GCV) — O(1) per alpha
# instead of O(n_folds) per alpha like KFold CV
# ============================================================

from sklearn.linear_model import RidgeCV

# Large search grid for alpha
alphas_search = np.logspace(-2, 5, 200)

# RidgeCV with LOOCV (cv=None uses GCV by default)
ridge_cv_model = RidgeCV(
    alphas=alphas_search,
    fit_intercept=True,
    scoring='neg_mean_squared_error',
    cv=5    # 5-fold instead of LOO for speed
)

# Scale X first to ensure fair regularisation
X_scaled_full = StandardScaler().fit_transform(X_raw)

ridge_cv_model.fit(X_scaled_full, y_raw)

print('RidgeCV results:')
print('=' * 50)
print(f'  Best alpha found:          {ridge_cv_model.alpha_:.4f}')
print(f'  Intercept:                 {ridge_cv_model.intercept_:.2f}')
print()

# Show top-5 most influential features
coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': ridge_cv_model.coef_,
    'Abs_Coef': np.abs(ridge_cv_model.coef_)
}).sort_values('Abs_Coef', ascending=False)

print('Feature importances (by |coefficient|):')
print(coef_df[['Feature', 'Coefficient']].to_string(index=False, float_format='{:.2f}'.format))

# Training prediction quality
y_pred_ridge = ridge_cv_model.predict(X_scaled_full)
train_mse    = mean_squared_error(y_raw, y_pred_ridge)
print(f'\nTraining RMSE: ${np.sqrt(train_mse):,.2f}')

In [ ]:
# ============================================================
# CELL 13: Predicted vs Actual — Diagnostic Plot
# ============================================================

# Fit final Ridge model with best alpha
final_ridge = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', Ridge(alpha=ridge_cv_model.alpha_, fit_intercept=True))
])
final_ridge.fit(X_raw, y_raw)
y_pred_final = final_ridge.predict(X_raw)

# OLS for comparison
final_ols = Pipeline([
    ('scaler', StandardScaler()),
    ('ols', LinearRegression())
])
final_ols.fit(X_raw, y_raw)
y_pred_ols = final_ols.predict(X_raw)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, y_pred, label, color in [
        (axes[0], y_pred_ols,   'OLS',   OLS_COLOR),
        (axes[1], y_pred_final, f'Ridge (α={ridge_cv_model.alpha_:.1f})', RIDGE_COLOR)]:

    ax.scatter(y_raw / 1e3, y_pred / 1e3, alpha=0.35, s=18, color=color)
    # Perfect prediction line
    lims = [min(y_raw.min(), y_pred.min()) / 1e3,
            max(y_raw.max(), y_pred.max()) / 1e3]
    ax.plot(lims, lims, 'k--', linewidth=1.5, label='Perfect prediction')
    rmse_val = np.sqrt(mean_squared_error(y_raw, y_pred))
    ax.set_title(f'{label}\nRMSE = ${rmse_val:,.0f}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Actual Price ($000s)', fontsize=10)
    ax.set_ylabel('Predicted Price ($000s)', fontsize=10)
    ax.legend(fontsize=9)

plt.suptitle('Predicted vs Actual House Price: OLS vs Ridge', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('ridge_predicted_vs_actual.png', dpi=120, bbox_inches='tight')
plt.show()
print('Note: On training data OLS often fits better — the real gain is on held-out test data.')

In [ ]:
# ============================================================
# CELL 14: Bias-Variance Trade-off Summary Plot
# Schematic showing how Ridge navigates bias vs variance
# ============================================================

# Simulate bias and variance components across alpha values
# using multiple train/test splits
alphas_bv     = np.logspace(-2, 4, 50)
n_trials      = 50
predictions   = np.zeros((n_trials, len(alphas_bv), 50))   # (trials, alphas, test_pts)

np.random.seed(0)
test_idx = np.random.choice(n_samples, size=50, replace=False)
train_idx = np.array([i for i in range(n_samples) if i not in test_idx])
X_test_bv = X_scaled[test_idx]
y_test_bv  = y_centred[test_idx]

for trial in range(n_trials):
    # Resample training data (simulates different datasets from same distribution)
    sample_idx = np.random.choice(train_idx, size=len(train_idx), replace=True)
    Xtr, ytr  = X_scaled[sample_idx], y_centred[sample_idx]
    for k, alpha in enumerate(alphas_bv):
        theta_k = ridge_analytical(Xtr, ytr, alpha=alpha)
        predictions[trial, k, :] = X_test_bv @ theta_k

# Bias^2: (mean prediction - true)^2, averaged over test points
mean_pred = predictions.mean(axis=0)          # (alphas, test_pts)
bias_sq   = ((mean_pred - y_test_bv) ** 2).mean(axis=1)
# Variance: variance of predictions across trials, averaged over test points
variance  = predictions.var(axis=0).mean(axis=1)
total_err = bias_sq + variance

fig, ax = plt.subplots(figsize=(10, 5))
ax.semilogx(alphas_bv, bias_sq    / 1e8, color='#E91E63', linewidth=2.2, label='Bias²')
ax.semilogx(alphas_bv, variance   / 1e8, color='#2196F3', linewidth=2.2, label='Variance')
ax.semilogx(alphas_bv, total_err  / 1e8, color='#4CAF50', linewidth=2.5, linestyle='--', label='Total Error')

best_bv = alphas_bv[np.argmin(total_err)]
ax.axvline(best_bv, color='black', linestyle=':', linewidth=1.5, label=f'Optimal α ≈ {best_bv:.2f}')

ax.set_xlabel('Regularisation Strength α (log scale)', fontsize=12)
ax.set_ylabel('Error Component (×10⁸)', fontsize=12)
ax.set_title('Bias-Variance Trade-off in Ridge Regression\n'
             'Optimal α balances bias (too much) vs variance (too little regularisation)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('ridge_bias_variance.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Optimal alpha (bias-variance sweet spot): {best_bv:.4f}')

In [ ]:
# ============================================================
# CELL 15: Comparison Summary Table
# OLS vs Ridge — key properties side by side
# ============================================================

summary_data = {
    'Property': [
        'Unique solution?',
        'Handles multicollinearity?',
        'Zeroes out coefficients?',
        'Closed-form solution?',
        'Hyperparameter?',
        'Bayesian interpretation',
        'Constraint geometry',
        'Best use case',
        'Sklearn class'
    ],
    'OLS (α=0)': [
        'Only if X has full rank',
        'No — unstable coefficients',
        'No',
        'Yes: θ=(XᵀX)⁻¹Xᵀy',
        'None',
        'Flat (uninformative) prior',
        'No constraint',
        'Low-dimensional, no collinearity',
        'LinearRegression'
    ],
    'Ridge (L2)': [
        'Always (even rank-deficient X)',
        'Yes — distributes weight evenly',
        'No — shrinks but not to zero',
        'Yes: θ=(XᵀX+αI)⁻¹Xᵀy',
        'α ≥ 0',
        'Gaussian prior N(0, σ²/α)',
        'L2 ball (circle)',
        'All features relevant, correlated features',
        'Ridge / RidgeCV'
    ]
}

summary_df = pd.DataFrame(summary_data)
print('OLS vs Ridge Regression — Summary')
print('=' * 80)
print(summary_df.to_string(index=False))

# Also show the final coefficient magnitudes
print('\n\nFinal Model (RidgeCV) — Coefficient Magnitudes:')
print('-' * 50)
for feat, coef in zip(feature_names, ridge_cv_model.coef_):
    bar = '█' * int(abs(coef) / 3000)
    sign = '+' if coef >= 0 else '-'
    print(f'  {feat:<22} {sign}{abs(coef):>8,.0f}  {bar}')

## Key Takeaways

| Concept | Plain-English Summary |
|---------|----------------------|
| **L2 penalty** | Adds $\alpha \sum \theta_j^2$ to the loss — penalises large coefficients quadratically |
| **Shrinkage** | All coefficients shrink toward zero proportionally — no exact zeros (unlike L1) |
| **Analytical solution** | $\theta = (X^\top X + \alpha I)^{-1} X^\top y$ — always exists, always unique |
| **Multicollinearity** | Ridge distributes weight evenly across correlated features; OLS goes wild |
| **Conditioning** | Adding $\alpha I$ reduces the condition number — more numerically stable |
| **Bayesian view** | Ridge = MAP with Gaussian prior: "I believe coefficients are near zero" |
| **Bias-variance** | Larger $\alpha$ = more bias + less variance; choose via cross-validation |
| **When to use** | All features are genuinely relevant (just need moderation), especially with correlated features |

---

## When NOT to Use Ridge

- You want **feature selection** (need some coefficients to be exactly zero) → use **Lasso**
- You have many truly **irrelevant features** → use **Lasso** or **ElasticNet**
- Features are uncorrelated and you have very few of them → plain OLS is fine
- You need a sparse, interpretable model → use **Lasso**

## Self-Check Questions

Answer these before looking at the solutions. Spend at least 2 minutes thinking through each one.

---

**Question 1:**  
OLS gives θ₁ = 50,000 and θ₂ = −48,000 for two highly correlated features (`sqft` and `num_rooms`).  
Ridge gives θ₁ = 8,000 and θ₂ = 6,000 for the same features.  
Which solution is more trustworthy, and why?

<details>
<summary><strong>Answer (click to reveal)</strong></summary>

**Ridge is more trustworthy.** Here is why:

When two features are highly correlated (say r = 0.95), the design matrix X is near-singular. The OLS solution is mathematically valid but practically meaningless — the large opposite-sign coefficients (50,000 and −48,000) are essentially cancelling each other out. They are telling you: "increase sqft by 1 and the price goes up by 50k, but that one sqft also adds roughly one room which decreases price by 48k." This is unstable: a tiny change in the data would swap the signs or double the magnitudes entirely (as we saw in the bootstrap experiment — standard deviation of thousands).

Ridge's output (8,000 and 6,000) makes physical sense: both sqft and num_rooms should have a **positive** effect on price, and both should be of similar magnitude since they carry overlapping information. Ridge achieves this by distributing the predictive weight evenly across the correlated group.

The key test: **bootstrap stability**. If you resample the data 200 times, the OLS coefficients will swing wildly. The Ridge coefficients will be stable. Stability = trustworthiness.

</details>

---

**Question 2:**  
Ridge regression always has a unique solution, while Lasso can have multiple solutions when features are correlated. Why?  
*(Hint: think about the shape of the objective function.)*

<details>
<summary><strong>Answer (click to reveal)</strong></summary>

The difference comes from **strict convexity** vs **convexity**.

**Ridge:** The L2 penalty $\alpha \sum \theta_j^2$ is **strictly convex** (the Hessian is positive definite: $X^\top X + \alpha I$ has all positive eigenvalues). A strictly convex function has **exactly one** minimum. This is why the analytical formula always gives a unique answer — even when XᵀX is singular.

**Lasso:** The L1 penalty $\alpha \sum |\theta_j|$ is **convex but NOT strictly convex** — it has flat regions (the absolute value function has zero second derivative almost everywhere). When two features are perfectly correlated, the Lasso objective has a **flat ridge** (a set of equally-good solutions), and the algorithm may land anywhere on it depending on initialisation or the solver. That's why Lasso arbitrarily picks one of two correlated features and zeros out the other — both choices are equally optimal.

**Geometric summary:** Ridge's L2 ball is smooth and rotationally symmetric; it always touches the MSE contours at exactly one point. The L1 diamond has flat faces; MSE contours can align with an entire face, creating a continuum of solutions.

</details>

---

**Question 3:**  
As α→∞, Ridge coefficients approach zero. But OLS has the lowest training error.  
So why would you ever choose Ridge over OLS?

<details>
<summary><strong>Answer (click to reveal)</strong></summary>

Because **lower training error does not mean better generalisation** — this is the core of the bias-variance tradeoff.

OLS minimises training MSE perfectly — it fits every quirk, noise fluctuation, and accidental correlation in the training set. On new (unseen) data, those quirks don't exist, so OLS predictions degrade. This is **overfitting**.

Ridge deliberately accepts a small increase in training error (bias) in exchange for a large reduction in coefficient instability (variance). The result is a model that:

1. **Generalises better** to new houses not seen during training
2. **Is more stable** — if you collected a second sample of 300 houses, the Ridge coefficients would be nearly the same; OLS coefficients could look completely different
3. **Is numerically safe** — OLS breaks down entirely when features are perfectly collinear; Ridge never does

The cross-validation experiment in Cell 10 demonstrates this directly: Ridge at the optimal α achieves **lower CV MSE** than OLS, even though OLS has lower training MSE.

Practical rule: if you care about **deployment** (real-world prediction accuracy), Ridge is usually safer than OLS in any dataset with more than ~10 features or any sign of multicollinearity.

</details>

## Worked Example: Ridge for a New House

Let's walk through a complete end-to-end prediction:

In [ ]:
# ============================================================
# CELL 16: End-to-End Prediction Example
# Predict price for a specific house using the fitted Ridge model
# ============================================================

# New house to value:
new_house = {
    'sqft':              2200,   # 2,200 sq ft
    'age':               10,     # 10 years old
    'lot_size':          7500,   # 7,500 sq ft lot
    'garage':            2,      # 2-car garage
    'bathrooms':         3,      # 3 bathrooms
    'stories':           2,      # 2 stories
    'pool':              0,      # no pool
    'school_rating':     8.5,    # good schools
    'crime_index':       2.0,    # low crime
    'distance_cbd':      8,      # 8 miles to city centre
    'num_rooms':         7,      # 7 rooms total
    'renovation_year':   2018,   # renovated 2018
    'zip_score':         4.0,
    'neighbourhood_price': 850,
    'month_listed':      6,      # June listing
    'agent_id':          12,
    'listing_photos':    22,
    'basement':          1,      # has basement
    'fireplace':         1,      # has fireplace
    'sqft_x_rooms':      2200 * 7 / 1000   # interaction term
}

# Convert to array (same feature order as training)
x_new = np.array([new_house[f] for f in feature_names]).reshape(1, -1)

# Predict using the pipeline (handles scaling internally)
predicted_price = final_ridge.predict(x_new)[0]
ols_price       = final_ols.predict(x_new)[0]

print('House Valuation Report')
print('=' * 45)
print(f'  Size:          {new_house["sqft"]:,} sqft')
print(f'  Age:           {new_house["age"]} years')
print(f'  Bedrooms:      ~{int(new_house["num_rooms"]-2)} (approx)')
print(f'  School rating: {new_house["school_rating"]}/10')
print(f'  Crime index:   {new_house["crime_index"]}/10 (low)')
print()
print(f'  OLS estimate:         ${ols_price:>12,.0f}')
print(f'  Ridge estimate:       ${predicted_price:>12,.0f}')
print(f'  Difference:           ${abs(predicted_price - ols_price):>12,.0f}')
print()
print(f'  Note: Ridge estimate is more reliable because the model')
print(f'  was trained with regularisation to avoid overfitting.')

In [ ]:
# ============================================================
# CELL 17: What Happens When You Perfectly Duplicate a Feature?
# Extreme multicollinearity demo — OLS completely breaks
# ============================================================

print('Extreme Collinearity Test')
print('=' * 50)

# Create a 2-feature dataset where feature 2 = feature 1 exactly
X_dup = np.column_stack([sqft, sqft])   # identical columns!
y_dup = y_centred

# Scale
X_dup_scaled = StandardScaler().fit_transform(X_dup)

print('\nOLS on perfectly duplicated features:')
try:
    # This may fail or give garbage
    theta_ols_dup = np.linalg.solve(X_dup_scaled.T @ X_dup_scaled,
                                    X_dup_scaled.T @ y_dup)
    print(f'  OLS theta: {theta_ols_dup}  ← arbitrary, unstable!')
except np.linalg.LinAlgError as e:
    print(f'  OLS FAILED: {e}')

# OLS via lstsq (uses SVD — handles rank-deficiency but still arbitrary)
theta_lstsq, _, rank, sv = np.linalg.lstsq(X_dup_scaled, y_dup, rcond=None)
print(f'  OLS lstsq: {theta_lstsq}  (rank={rank})')
print(f'  Singular values: {sv.round(2)}')

print('\nRidge on perfectly duplicated features:')
for alpha_dup in [1.0, 10.0, 100.0]:
    theta_r = ridge_analytical(X_dup_scaled, y_dup, alpha=alpha_dup)
    print(f'  alpha={alpha_dup:>6}: theta = [{theta_r[0]:>10.2f}, {theta_r[1]:>10.2f}]  '
          f'(sum={theta_r.sum():.2f}, symmetric!)')

print()
print('Key insight: With perfect collinearity, OLS is arbitrary.')
print('Ridge always finds the symmetric solution (θ₁ = θ₂ = OLS_θ/2).')
print('This is the "fairest" answer: equal credit to identical features.')

## Next Steps

You have now mastered Ridge regression. Here is what to explore next:

1. **Notebook 04 — ElasticNet:** Combines L1 (feature selection) and L2 (coefficient moderation) in one model. Best of both worlds — especially for correlated feature groups.

2. **Comparison exercise:** Take the same house price dataset and compare:
   - OLS: how many features have |coef| > 10,000?
   - Ridge: same check — are they more uniform?
   - Lasso: which features get zeroed out?

3. **Real data practice:** Try `sklearn.datasets.fetch_california_housing()` and apply Ridge with RidgeCV.

4. **Research question:** What is the relationship between Ridge regression and Principal Component Regression (PCR)? Both handle multicollinearity — how do their solutions differ?

---

**Formula Reference Card:**

| Formula | Meaning |
|---------|--------|
| $J(\theta) = \frac{1}{2n}\|y - X\theta\|^2 + \alpha\|\theta\|^2$ | Ridge objective |
| $\theta^* = (X^\top X + \alpha I)^{-1} X^\top y$ | Analytical solution |
| $\kappa(X^\top X + \alpha I) < \kappa(X^\top X)$ | Improved conditioning |
| $\theta_j \sim \mathcal{N}(0, \sigma^2/\alpha)$ | Bayesian prior |
| `Ridge(alpha=1.0, fit_intercept=True)` | sklearn API |
| `RidgeCV(alphas=[0.1, 1, 10])` | with CV selection |